# Mochiichao — ComfyUI + Wan 2.2 (Kaggle GPU endpoint) — v2 ACCELERATED

The pipeline's generative-pass notebook (UNIFIED-PIPELINE triple-lock stack). Runs on **GPU T4 x2** (set Accelerator before running). Quota discipline: ~22h/week free GPU — batch jobs, don't idle the session.

**v2 adds the Wan acceleration stack** (researched + verified 2026-10-27; full write-up in TOOLS-MANIFEST.md §Wan acceleration stack):
- **Step distillation is the big lever.** There is NO LightX2V/Lightning 4-step LoRA for TI2V-5B (verified: lightx2v/Wan2.2-Lightning ships A14B high/low pairs only). The 5B equivalent is the **Wan22 Turbo LoRA** (Self-Forcing distill, extracted by Kijai): 20 steps × CFG 5.5 → **4 steps × CFG 1.0**.
- **TeaCache/MagCache for Wan 2.2 live in Kijai's WanVideoWrapper** (the standalone ComfyUI-TeaCache pack stops at Wan 2.1). Wired into the finals-path cell at rel_l1_thresh 0.15.
- **torch.compile** (inductor) via the wrapper's compile-settings node — expect a multi-minute first-run compile; the A/B protocol times a WARM run.
- **SageAttention / FlashAttention 2 do NOT run on T4 (sm75)** — Ampere+ only. On T4, attention = PyTorch SDPA. Do not pass --use-sage-attention / --use-flash-attention.
- **GGUF Q8_0** is staged as a VRAM fallback (the fp16 5B already fits a 15GB T4; GGUF buys headroom, not raw speed).

Flow: GPU check → ComfyUI install → Wan 2.2 weights → likeness-lock + acceleration nodes → acceleration weights → headless API server → smoke test → **A/B timed runs (baseline vs optimized, same seed/prompt)**.


In [ ]:
# 1. GPU check
!nvidia-smi
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')

In [ ]:
# 2. ComfyUI (pinned shallow clone) + requirements
import os
os.chdir('/kaggle/working')
if not os.path.exists('ComfyUI'):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git
os.chdir('ComfyUI')
!pip install -q -r requirements.txt
print('ComfyUI ready')

In [ ]:
# 3. Wan 2.2 weights - VERIFIED file list (Comfy-Org/Wan_2.2_ComfyUI_Repackaged, checked 2026-10-27)
# TI2V-5B fp16 (image-to-video, fits T4 x2) + umt5 fp8 text encoder + Wan 2.2 VAE.
# Staging rule: masters live on TeraBox; this pulls the session working set.
from huggingface_hub import hf_hub_download
import shutil, os
REPO = 'Comfy-Org/Wan_2.2_ComfyUI_Repackaged'
FILES = {
 'split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors': 'models/diffusion_models',
 'split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors': 'models/text_encoders',
 'split_files/vae/wan2.2_vae.safetensors': 'models/vae',
}
for src, dst in FILES.items():
    path = hf_hub_download(repo_id=REPO, filename=src)
    os.makedirs(dst, exist_ok=True)
    shutil.copy(path, os.path.join(dst, os.path.basename(src)))
    print('staged', src, '->', dst)
print('weights ready')


In [ ]:
# 3b. Acceleration weights (all free; licences in TOOLS-MANIFEST)
# - Wan22 Turbo LoRA (Self-Forcing step distill for TI2V-5B, via Kijai):
#   THE 4-step lever for the 5B. LightX2V has no 5B LoRA — verified.
# - FastWan FullAttn checkpoint (alternative 3-step distilled full model, Apache-2.0).
# - GGUF Q8_0 (optional VRAM fallback; speed-neutral on T4).
from huggingface_hub import hf_hub_download
import shutil, os
os.chdir('/kaggle/working/ComfyUI')
JOBS = [
 ('Kijai/WanVideo_comfy', 'LoRAs/Wan22-Turbo/Wan22_TI2V_5B_Turbo_lora_rank_64_fp16.safetensors', 'models/loras'),
 ('Kijai/WanVideo_comfy', 'FastWan/Wan2_2-TI2V-5B-FastWanFullAttn_bf16.safetensors', 'models/diffusion_models'),
]
DOWNLOAD_GGUF = True
if DOWNLOAD_GGUF:
    JOBS.append(('QuantStack/Wan2.2-TI2V-5B-GGUF', 'Wan2.2-TI2V-5B-Q8_0.gguf', 'models/unet'))
for repo, src, dst in JOBS:
    path = hf_hub_download(repo_id=repo, filename=src)
    os.makedirs(dst, exist_ok=True)
    shutil.copy(path, os.path.join(dst, os.path.basename(src)))
    print('staged', src, '->', dst)
print('acceleration weights ready')


In [ ]:
# 4. Custom nodes: likeness lock (triple lock) + acceleration stack
import os
os.chdir('/kaggle/working/ComfyUI/custom_nodes')
for repo in ['https://github.com/ltdrdata/ComfyUI-Manager.git',
             'https://github.com/cubiq/ComfyUI_IPAdapter_plus.git',
             'https://github.com/Fannovel16/comfyui_controlnet_aux.git',
             # Acceleration: Kijai wrapper = Wan 2.2-native TeaCache/MagCache,
             # block swap, torch.compile settings, Turbo/FastWan examples.
             'https://github.com/kijai/ComfyUI-WanVideoWrapper.git',
             # GGUF loader (VRAM fallback path for the 5B).
             'https://github.com/city96/ComfyUI-GGUF.git']:
    d=repo.split('/')[-1].replace('.git','')
    if not os.path.exists(d):
        os.system(f'git clone --depth 1 {repo}')
!pip install -q -r /kaggle/working/ComfyUI/custom_nodes/ComfyUI-WanVideoWrapper/requirements.txt
!pip install -q gguf
# NOTE: standalone ComfyUI-TeaCache is NOT installed: its Wan support stops
# at Wan 2.1 (no Wan 2.2 coefficients). Wan 2.2 caching rides the wrapper.
print('nodes staged — character LoRAs trained on the owner\'s own art drop into models/loras/ per scene batch')


In [ ]:
# 5. Launch the headless API server (background) — the pipeline drives ComfyUI through this API
import subprocess, time, os
os.chdir('/kaggle/working/ComfyUI')
proc = subprocess.Popen(['python','main.py','--listen','127.0.0.1','--port','8188'], stdout=open('/kaggle/working/comfy.log','w'), stderr=subprocess.STDOUT)
time.sleep(20)
print('server pid', proc.pid)

In [ ]:
# 6. Smoke test: queue an empty/default workflow ping — a real scene workflow posts the same way
import urllib.request, json
try:
    r = urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=10)
    print('API live:', r.status)
except Exception as e:
    print('API not up yet — check /kaggle/working/comfy.log; models may still be loading.', e)

In [ ]:
# 7. A/B timing harness — shared constants (single source of truth for BOTH runs)
# A/B validity rule: baseline and optimized use IDENTICAL prompt strings, seed,
# frame count and latent size. If the original test session is still alive, pull
# its executed graph from /history (see README-kaggle.md audit steps) and paste
# the original prompt strings over these defaults BEFORE the timed runs.
import json, time, uuid, urllib.request
try:
    import websocket  # websocket-client
except ImportError:
    !pip install -q websocket-client
    import websocket

SERVER = '127.0.0.1:8188'
IMAGE_NAME = 'wispkin.png'   # must exist in ComfyUI/input (uploaded in test 1)
SEED = 42
WIDTH, HEIGHT, LENGTH, FPS = 480, 1024, 49, 16
POS = ("Animate THIS EXACT character image, preserving its design and flat 2D cartoon style precisely. "
       "The character is a small round black creature holding a white cracked skull mask onto its face with two "
       "small black hands, smoke trailing beneath it. It floats gently up and down in place, the mask tilting "
       "slowly side to side as if it is listening, smoke curling softly below. Do not change the character design, "
       "do not add anything, no 3D rendering, no new characters. Locked-off camera, plain solid background.")
NEG = ("3d render, cgi, pixar, photorealistic, realistic, blurry, distorted, deformed, morphing, changing design, "
       "extra characters, extra limbs, text, watermark, logo, camera movement, zoom")

def run_timed(workflow, label):
    """Queue a workflow via the API; time it from websocket executing events."""
    client_id = uuid.uuid4().hex
    ws = websocket.create_connection(f'ws://{SERVER}/ws?clientId={client_id}')
    body = json.dumps({'prompt': workflow, 'client_id': client_id}).encode()
    req = urllib.request.Request(f'http://{SERVER}/prompt', data=body,
                                 headers={'Content-Type': 'application/json'})
    pid = json.loads(urllib.request.urlopen(req).read())['prompt_id']
    node_start, marks, t0 = {}, {}, None
    while True:
        msg = json.loads(ws.recv())
        if msg['type'] == 'executing' and msg['data'].get('prompt_id') == pid:
            node = msg['data'].get('node')
            now = time.time()
            if node is None:
                total = now - t0 if t0 else None
                break
            if t0 is None: t0 = now
            node_start[node] = now
            marks[node] = workflow[node]['class_type']
        elif msg['type'] == 'executed' and msg['data'].get('prompt_id') == pid:
            n = msg['data']['node']
            if n in node_start:
                print(f'  [{label}] {marks.get(n, n):28s} {time.time() - node_start[n]:8.1f}s')
        elif msg['type'] == 'execution_error' and msg['data'].get('prompt_id') == pid:
            print(f'  [{label}] EXECUTION ERROR:', json.dumps(msg['data'])[:800]); total = None; break
    ws.close()
    print(f'== [{label}] TOTAL {total:.1f}s' if total else f'== [{label}] FAILED')
    return total

def graph_doctor(*classes):
    """Print required inputs for node classes — use to fix API graphs if the
    wrapper's input names drift between versions."""
    info = json.loads(urllib.request.urlopen(f'http://{SERVER}/object_info').read())
    for c in classes:
        spec = info.get(c, {}).get('input', {})
        print(c, '-> required:', list(spec.get('required', {}).keys()),
              '| optional:', list(spec.get('optional', {}).keys()))
    return info

def audit_graph(prompt_id=None):
    """2x-resolution audit: dump an executed graph's node classes, flag any
    upscale stage, and report the latent size implied by WanImageToVideo +
    the 16x-stride Wan 2.2 VAE."""
    hist = json.loads(urllib.request.urlopen(f'http://{SERVER}/history').read())
    pid = prompt_id or list(hist.keys())[-1]
    graph = hist[pid]['prompt'][2]
    print('nodes:', sorted({n['class_type'] for n in graph.values()}))
    for nid, n in graph.items():
        if 'pscale' in n['class_type']:
            print('!! UPSCALE NODE PRESENT:', nid, n['class_type'], n.get('inputs'))
        if n['class_type'] in ('WanImageToVideo', 'Wan22ImageToVideoLatent'):
            w, h = n['inputs'].get('width'), n['inputs'].get('height')
            print(f'{n["class_type"]}: width={w} height={h} -> pixels at 16x stride = {w}x{h}; '
                  f'latent would be {w//16}x{h//16}. If latent was instead built at /8, output doubles.')
    return graph


In [ ]:
# 8. A/B RUN A — BASELINE (exact replica of the measured test-1 settings)
# Native nodes, stock fp16 5B, 20 steps, uni_pc, CFG 5.5, seed 42, 480x1024 latent
# intent, NO upscale anywhere. Reference as-run time (2026-10-27): 1708s (28m28s).
def baseline_workflow():
    return {
      "1": {"class_type": "UNETLoader", "inputs": {"unet_name": "wan2.2_ti2v_5B_fp16.safetensors", "weight_dtype": "default"}},
      "2": {"class_type": "CLIPLoader", "inputs": {"clip_name": "umt5_xxl_fp8_e4m3fn_scaled.safetensors", "type": "wan", "device": "default"}},
      "3": {"class_type": "VAELoader", "inputs": {"vae_name": "wan2.2_vae.safetensors"}},
      "4": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["2", 0], "text": POS}},
      "5": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["2", 0], "text": NEG}},
      "6": {"class_type": "LoadImage", "inputs": {"image": IMAGE_NAME}},
      "7": {"class_type": "WanImageToVideo", "inputs": {"vae": ["3", 0], "width": WIDTH, "height": HEIGHT, "length": LENGTH, "batch_size": 1, "positive": ["4", 0], "negative": ["5", 0], "start_image": ["6", 0]}},
      "8": {"class_type": "ModelSamplingSD3", "inputs": {"model": ["1", 0], "shift": 5.0}},
      "9": {"class_type": "KSampler", "inputs": {"model": ["8", 0], "seed": SEED, "steps": 20, "cfg": 5.5, "sampler_name": "uni_pc", "scheduler": "simple", "positive": ["7", 0], "negative": ["7", 1], "latent_image": ["7", 2], "denoise": 1.0}},
      "10": {"class_type": "VAEDecode", "inputs": {"vae": ["3", 0], "samples": ["9", 0]}},
      "11": {"class_type": "CreateVideo", "inputs": {"images": ["10", 0], "fps": float(FPS)}},
      "12": {"class_type": "SaveVideo", "inputs": {"video": ["11", 0], "filename_prefix": "wispkin_wan_baseline", "format": "auto", "codec": "auto"}},
    }
t_baseline = run_timed(baseline_workflow(), 'BASELINE')


In [ ]:
# 9. A/B RUN B — OPTIMIZED (the acceleration stack)
# Kijai wrapper path (Wan 2.2-native accel nodes) + Wan22 Turbo LoRA:
# 4 steps, CFG 1.0, unipc, shift 5.0, seed 42, latent exactly 480x1024,
# torch.compile settings ON, NO TeaCache (pointless on a 4-step trajectory),
# NO upscale anywhere. Run this cell TWICE: run 1 pays the torch.compile
# warmup (minutes — not a hang); run 2 is the timed warm run.
# If validation errors name wrapper inputs, run graph_doctor('WanVideoSampler',
# 'WanVideoModelLoader', 'WanVideoLoraSelect') and/or diff against the Turbo
# example workflow shipped in ComfyUI-WanVideoWrapper/example_workflows/.
def optimized_workflow():
    return {
      "1": {"class_type": "WanVideoTorchCompileSettings", "inputs": {"backend": "inductor", "fullgraph": False, "mode": "default", "dynamic": False, "dynamo_cache_size_limit": 64, "compile_transformer_blocks_only": True}},
      "2": {"class_type": "WanVideoLoraSelect", "inputs": {"lora": "Wan22_TI2V_5B_Turbo_lora_rank_64_fp16.safetensors", "strength": 1.0, "low_mem_load": False, "merge_loras": False}},
      "3": {"class_type": "WanVideoModelLoader", "inputs": {"model": "wan2.2_ti2v_5B_fp16.safetensors", "base_precision": "fp16", "quantization": "disabled", "load_device": "main_device", "attention_mode": "sdpa", "compile_args": ["1", 0], "lora_args": ["2", 0]}},
      "4": {"class_type": "WanVideoVAELoader", "inputs": {"model_name": "wan2.2_vae.safetensors", "precision": "fp16"}},
      "5": {"class_type": "WanVideoTextEncode", "inputs": {"model": "umt5_xxl_fp8_e4m3fn_scaled.safetensors", "precision": "fp16", "positive_prompt": POS, "negative_prompt": NEG, "quantization": "disabled", "load_device": "main_device", "force_offload": False}},
      "6": {"class_type": "LoadImage", "inputs": {"image": IMAGE_NAME}},
      "7": {"class_type": "WanVideoImageToVideoEncode", "inputs": {"vae": ["4", 0], "width": WIDTH, "height": HEIGHT, "num_frames": LENGTH, "start_image": ["6", 0], "noise_aug_strength": 0.0, "start_latent_strength": 1.0, "end_latent_strength": 1.0}},
      "8": {"class_type": "WanVideoSampler", "inputs": {"model": ["3", 0], "image_embeds": ["7", 0], "text_embeds": ["5", 0], "steps": 4, "cfg": 1.0, "shift": 5.0, "seed": SEED, "force_offload": True, "scheduler": "unipc", "riflex_freq_index": 0, "denoise_strength": 1.0, "batched_cfg": False}},
      "9": {"class_type": "WanVideoDecode", "inputs": {"vae": ["4", 0], "samples": ["8", 0], "enable_vae_tiling": True, "tile_x": 272, "tile_y": 272, "tile_stride_x": 144, "tile_stride_y": 128, "normalization": "default"}},
      "10": {"class_type": "CreateVideo", "inputs": {"images": ["9", 0], "fps": float(FPS)}},
      "11": {"class_type": "SaveVideo", "inputs": {"video": ["10", 0], "filename_prefix": "wispkin_wan_optimized", "format": "auto", "codec": "auto"}},
    }
print('warmup run (pays torch.compile cost, untimed):')
run_timed(optimized_workflow(), 'OPTIMIZED-WARMUP')
t_optimized = run_timed(optimized_workflow(), 'OPTIMIZED')
if t_baseline and t_optimized:
    print(f'A/B: baseline {t_baseline:.0f}s vs optimized {t_optimized:.0f}s = {t_baseline/t_optimized:.1f}x')
    print('Replace the community-reported estimates in TOOLS-MANIFEST/README with these measured numbers.')


In [ ]:
# 10. OPTIONAL — FINALS PATH (quality renders, still accelerated)
# Stock model at full 20 steps / CFG 5.5, but with WanVideoTeaCache
# (rel_l1_thresh 0.15 — inside the researched Wan I2V band 0.1-0.3) and
# torch.compile. This is the path for keeper renders once prompts are locked;
# expect ~1.5-2x vs baseline (community-reported until measured here).
RUN_FINALS_PATH = False
def finals_workflow():
    w = optimized_workflow()
    del w["2"]  # no Turbo LoRA on the finals path
    w["3"] = {"class_type": "WanVideoModelLoader", "inputs": {"model": "wan2.2_ti2v_5B_fp16.safetensors", "base_precision": "fp16", "quantization": "disabled", "load_device": "main_device", "attention_mode": "sdpa", "compile_args": ["1", 0]}}
    w["12"] = {"class_type": "WanVideoTeaCache", "inputs": {"rel_l1_thresh": 0.15, "start_step": 0, "end_step": -1, "cache_device": "main_device"}}
    w["8"]["inputs"].update({"steps": 20, "cfg": 5.5, "cache_args": ["12", 0]})
    w["11"]["inputs"]["filename_prefix"] = "wispkin_wan_finals_teacache"
    return w
if RUN_FINALS_PATH:
    t_finals = run_timed(finals_workflow(), 'FINALS+TEACACHE')
